In [5]:
import pandas as pd
import numpy as np

np.random.seed(3)

# ---- 1. Cargar datos ----
df = pd.read_csv('winequality-red.csv', sep=';')
print(df.columns.tolist())
print(df['quality'].value_counts().sort_index())  # para ver el desbalance de clases

# ---- 2. Features y target ----
features = [c for c in df.columns if c != 'quality']
X_raw = df[features].values

clases = sorted(df['quality'].unique())  # ej: [3,4,5,6,7,8]
n_clases = len(clases)
print(f"\nClases encontradas: {clases}")

# One-hot
D_raw = np.zeros((len(df), n_clases))
for i, q in enumerate(df['quality'].values):
    D_raw[i, clases.index(q)] = 1

# ---- 3. Normalizar ----
mean = X_raw.mean(axis=0)
std = X_raw.std(axis=0)
X_norm = (X_raw - mean) / std

# ---- 4. Split ----
np.random.seed(42)
idx = np.random.permutation(len(X_norm))
split = int(0.8 * len(X_norm))
train_idx, test_idx = idx[:split], idx[split:]
X_train, D_train = X_norm[train_idx], D_raw[train_idx]
X_test, D_test = X_norm[test_idx], D_raw[test_idx]

# ---- 5. Activaciones ----
def sigmoid(v):
    return 1 / (1 + np.exp(-v))

def softmax(v):
    ex = np.exp(v - np.max(v))
    return ex / np.sum(ex)

# ---- 6. Pesos ----
n_features = X_train.shape[1]
n_hidden = 12
W1 = 2 * np.random.rand(n_hidden, n_features) - 1
W2 = 2 * np.random.rand(n_clases, n_hidden) - 1
alpha = 0.3

# ---- 7. Entrenamiento ----
def train_step(W1, W2, X, D):
    for k in range(len(X)):
        x = X[k].reshape(-1, 1)
        d = D[k].reshape(-1, 1)

        v1 = W1 @ x
        y1 = sigmoid(v1)
        v = W2 @ y1
        y = softmax(v)

        e = d - y
        delta = e

        e1 = W2.T @ delta
        delta1 = y1 * (1 - y1) * e1

        W1 = W1 + alpha * (delta1 @ x.T)
        W2 = W2 + alpha * (delta @ y1.T)
    return W1, W2

for epoch in range(2000):
    W1, W2 = train_step(W1, W2, X_train, D_train)
    if epoch % 500 == 0:
        print(f"Época {epoch} completada")

# ---- 8. Evaluar ----
correctas = 0
for k in range(len(X_test)):
    x = X_test[k].reshape(-1, 1)
    v1 = W1 @ x
    y1 = sigmoid(v1)
    v = W2 @ y1
    y = softmax(v)
    pred = clases[np.argmax(y)]
    real = clases[np.argmax(D_test[k])]
    if pred == real:
        correctas += 1

print(f"\nAccuracy en test: {correctas/len(X_test)*100:.1f}%")

['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates', 'alcohol', 'quality']
quality
3     10
4     53
5    681
6    638
7    199
8     18
Name: count, dtype: int64

Clases encontradas: [np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
Época 0 completada
Época 500 completada
Época 1000 completada
Época 1500 completada

Accuracy en test: 50.3%
